# Quantum Ensemble Learning Tutorial

## Overview

This notebook demonstrates **quantum ensemble learning** for classification tasks using quantum cosine similarity classifiers. The approach combines multiple quantum classifiers through superposition to improve prediction accuracy and robustness.

### Key Concepts

1. **Quantum Cosine Classifier**: Measures similarity between quantum-encoded data points
2. **Quantum Ensemble**: Creates superpositions of different training data arrangements
3. **Random Unitary Ensemble**: Uses Haar-random unitaries for more general transformations

### References

- **This work**: [arXiv:2506.02213](https://arxiv.org/abs/2506.02213)
- **Original ensemble work**: Macaluso et al. [IET Quantum Communication](https://ietresearch.onlinelibrary.wiley.com/doi/full/10.1049/qtc2.12087)
- **Original Code repository**: [GitHub](https://github.com/amacaluso/Quantum-algorithm-for-ensemble-using-bagging)


In [ ]:
%load_ext autoreload
%autoreload 2

import sys
import os
import re
import numpy as np
import pandas as pd
import pickle

seed = 54321
n_shots = 8192
np.random.seed(seed)


## 1. Setup and Initialization

### Import Libraries and Set Paths

We begin by setting up the environment, importing necessary libraries, and defining output directories for storing results.


In [ ]:
dir_home = '.'
dir_output = os.path.join( dir_home, 'experiments' )

os.mkdir(dir_output) if not os.path.exists(dir_output) else None

file_predictions = os.path.join( dir_output, 'predictions.pkl' )
file_datasets = os.path.join( dir_output, 'datasets.pkl' )


from Utils import *
from modeling import *

## 2. Configuration Parameters

### Define Experiment Parameters

- **test_size**: Fraction of data used for testing (0.2 = 20%)
- **n_splits**: Number of cross-validation splits
- **seed**: Random seed for reproducibility
- **n_shots**: Number of quantum measurements per circuit

We also initialize storage objects for datasets and predictions.


In [ ]:

test_size = 0.2
n_splits = 10


# Object to hold all the datasets
if os.path.exists( file_datasets ):
    dataset = pickle.load( open( file_datasets, 'rb' ) )
else:
    dataset = {}
    
# Object to hold all the prediction results
if os.path.exists( file_predictions ):
    predictions = pickle.load( open( file_predictions, 'rb' ) )
else:
    predictions = {}
    

## 3. Dataset Generation

### Create Synthetic Blob Datasets

We generate synthetic datasets using scikit-learn's `make_blobs` function. These datasets consist of:
- **2D feature space**: Easy to visualize
- **2 classes**: Binary classification problem
- **Varying parameters**: Different cluster centers, standard deviations, and sample sizes

This allows us to test the quantum ensemble methods on controlled, well-understood data.


### Generate Multiple Dataset Variants

We create several blob datasets with different characteristics:
- Varying cluster separations
- Different noise levels (cluster_std)
- Multiple sample sizes

Each dataset is split into training and test sets for evaluation.


In [ ]:
from sklearn import datasets
from sklearn.model_selection import train_test_split
dataset_name = 'blob'
rerun = True

if (rerun) or (dataset_name not in dataset.keys()):
    predictions[dataset_name] = {}
    dataset[dataset_name] = {}
    for n_size in [100]:
        for std in [0.3, 0.5]:
            for p1 in [ 0.3, 0.5, 1]:
                for p2 in [ 0.3, 0.5, 1]:
                    centers=[[p1, p2],[p2, p1]]
                    for split in range(n_splits):
                        X, y = datasets.make_blobs(n_samples=n_size, centers=centers,
                                                    n_features=2, center_box=(0, 1),
                                                    cluster_std=std, random_state=seed)
                        X_train, X_test, y_train, y_test = train_test_split(X, y, stratify=y, random_state=seed, test_size=test_size)
                        
                        dataset[dataset_name][(split, n_size, std, p1, p2)] = (pd.DataFrame(X_train), 
                                                                               pd.DataFrame(X_test), 
                                                                               pd.Series(y_train), 
                                                                               pd.Series(y_test))

    pickle.dump( dataset, open( file_datasets, 'wb') )

## 4. Quantum Cosine Classifier

### Single Quantum Classifier Baseline

The quantum cosine classifier measures similarity between quantum states using:

1. **State Preparation**: Encode classical data as quantum states
2. **Controlled-SWAP Test**: Measure overlap between training and test states
3. **Hadamard Interference**: Extract similarity information
4. **Measurement**: Obtain classification probability

**Key Parameters**:
- `n_train`: Number of training samples (typically 1 for single classifier)
- `n_features`: Number of features (must be power of 2)
- `n_shots`: Measurement shots for probability estimation

This serves as a baseline for comparison with ensemble methods.


In [ ]:
method = 'qcosine'
rerun = False
n_trains = [1,2,4]
n_features = [2]

if (rerun) or (dataset_name not in predictions.keys()) or (method not in predictions[dataset_name].keys()):
    predictions = run_quantum_cosine(predictions, dataset, method, dataset_name, seed, test_size, file_predictions, 
                                     n_features, n_trains, n_shots)


## 5. Quantum Ensemble Method

### Ensemble with Controlled Swaps

The quantum ensemble creates superpositions of different training data arrangements using controlled swap operations.

**Algorithm**:
1. Initialize control qubits in superposition (Hadamard gates)
2. Apply controlled swaps to rearrange training data
3. Perform cosine similarity test
4. Measure to obtain ensemble prediction

**Key Parameters**:
- `d`: Number of control qubits (ensemble depth) - creates 2^d ensemble members
- `n_swap`: Number of swap operations per control qubit
- `n_train`: Number of training samples (must be even for balanced mode)
- `mode`: Sampling strategy
  - `"balanced"`: Class-balanced sampling
  - `"unbalanced"`: Random sampling
  - `"pair_sample"`: All pairwise swaps

**Expected Improvement**: Ensemble methods typically outperform single classifiers by:
- Reducing variance through averaging
- Exploring multiple data arrangements
- Leveraging quantum superposition


In [ ]:
method = 'qensemble' 
rerun = False
ds = [1,2,3]
n_trains = [2,4]
n_swaps = [1,2,4]
n_features = [2]
pca_embed = False
umap_embed = False
device = 'CPU'

if (rerun) or (dataset_name not in predictions.keys()) or (method not in predictions[dataset_name].keys()):
    predictions = run_quantum_ensemble(predictions, dataset, method, dataset_name, seed, test_size, file_predictions, 
                                     ds, n_swaps, n_features, n_trains, n_shots, pca_embed = pca_embed, umap_embed = umap_embed, device = device)


## 6. Random Unitary Ensemble

### Ensemble with Haar-Random Unitaries

This advanced variant uses random unitary transformations sampled from the Haar measure instead of fixed swap patterns.

**Advantages**:
- More general transformations of training data
- Potentially better exploration of hypothesis space
- Theoretical connections to quantum advantage

**Implementation**:
- Sample unitaries from `scipy.stats.unitary_group`
- Apply as controlled operations on data+label registers
- Creates richer superpositions than simple swaps

**Trade-offs**:
- Higher circuit depth and complexity
- More difficult to implement on real quantum hardware
- Potentially better generalization

**Note**: This method is more computationally intensive but may provide better results on complex datasets.


### Execute Random Unitary Ensemble

Run the advanced ensemble with Haar-random unitaries:
- Uses `modeling_random_unitary.py` module
- Applies random unitary transformations instead of swaps
- More general but computationally intensive

**Comparison**: Compare results with standard ensemble to see if random unitaries provide:
- Better accuracy
- Lower Brier score (better calibration)
- Improved generalization

**Warning**: This method requires significantly more computation time.


In [ ]:
method = 'qensemble_random_unitary'
rerun = False
ds = [1,2,3]
n_trains = [2,4]
n_swaps = [1,2,4]
n_features = [2]
pca_embed = False
umap_embed = False
device = 'CPU'

if (rerun) or (dataset_name not in predictions.keys()) or (method not in predictions[dataset_name].keys()):
    predictions = run_quantum_ensemble(predictions, dataset, method, dataset_name, seed, test_size, file_predictions, 
                                     ds, n_swaps, n_features, n_trains, n_shots, pca_embed = pca_embed, umap_embed = umap_embed, device = device,
                                     random_unitary=True)


## Conclusion

This tutorial demonstrated quantum ensemble learning for classification:

### Key Takeaways

1. **Quantum Advantage**: Ensemble methods leverage quantum superposition to evaluate multiple classifiers simultaneously
2. **Parameter Tuning**: Performance depends on careful selection of d, n_swap, and n_train
3. **Trade-offs**: Balance between accuracy improvement and computational cost
4. **Scalability**: Current methods limited by qubit count (~30-36 for simulation)

### Next Steps

- Test on real quantum hardware (IBM Quantum, etc.)
- Apply to real-world datasets
- Explore error mitigation techniques
- Investigate theoretical quantum advantage

### Further Reading

- See `README.md` for detailed documentation
- Check `modeling.py` and `modeling_random_unitary.py` for implementation details
- Refer to cited papers for theoretical background
